In [58]:
import zipfile, os, re
import numpy as np
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [59]:
zipfile.ZipFile("/content/archive.zip").extractall("classdataset")
path = "classdataset"
txt_path = os.path.join(path, os.listdir(path)[0])

In [60]:
with open(txt_path, 'r', encoding='utf-8-sig') as f:
    text = f.read()

In [74]:
start = text.find("*** START OF")
end = text.find("*** END OF")
if start != -1 and end != -1:
    text = text[text.find("\n", start):end]
text = text.lower()

In [62]:
sentences = sent_tokenize(text)
clean_sents = []
for s in sentences:
    s = re.sub(r'[^a-z\s]', ' ', s)
    words = s.split()
    if len(words) > 2:
        clean_sents.append(' '.join(words))
print("Total sentences:", len(clean_sents))

Total sentences: 4592


In [63]:
from tensorflow.keras.preprocessing.text import Tokenizer
tok = Tokenizer()
tok.fit_on_texts(clean_sents)
seq = tok.texts_to_sequences(clean_sents)
vocab_size = len(tok.word_index) + 1
print("Vocab size:", vocab_size)

Vocab size: 7821


In [64]:
MAX_LEN = 30
data = []
for s in seq:
    s = s[:MAX_LEN]
    for i in range(2, len(s) + 1):
        data.append(s[:i])

x = [r[:-1] for r in data]
y = [r[-1] for r in data]

In [65]:
from tensorflow.keras.utils import pad_sequences
max_len = max(len(i) for i in x)
x = pad_sequences(x, maxlen=max_len, padding='pre')
y = np.array(y)
print(x.shape, y.shape)

(85048, 29) (85048,)


In [66]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.23, random_state=42
)

In [67]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [68]:
model = Sequential([
    Embedding(vocab_size, 100, input_length=max_len),
    LSTM(150),
    Dropout(0.2),
    Dense(vocab_size, activation='softmax')
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [69]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam', metrics=['accuracy'])

In [70]:
model.fit(X_train, y_train, epochs=20, batch_size=128,validation_data=(X_test, y_test))


Epoch 1/20
512/512 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.0523 - loss: 6.5653 - val_accuracy: 0.0619 - val_loss: 6.3546
Epoch 2/20
512/512 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.0727 - loss: 6.1035 - val_accuracy: 0.0883 - val_loss: 6.1251
Epoch 3/20
512/512 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.1001 - loss: 5.7847 - val_accuracy: 0.1082 - val_loss: 5.9360
Epoch 4/20
512/512 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.1161 - loss: 5.5467 - val_accuracy: 0.1175 - val_loss: 5.8618
Epoch 5/20
512/512 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.1277 - loss: 5.3695 - val_accuracy: 0.1246 - val_loss: 5.8230
Epoch 6/20
512/512 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.1378 - loss: 5.2195 - val_accuracy: 0.1304 - val_loss: 5.8035
Epoch 7/20
512/512 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.1465 - loss: 5.0853 - val_accuracy: 0.1325 - val_loss: 5.7942
Epoch 8/20
512/512 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.1537 - loss: 4.9568 - val_accu

In [71]:
def predict_next(seed, n_words=5):
    for _ in range(n_words):
        t = tok.texts_to_sequences([seed.lower()])[0]
        t = pad_sequences([t], maxlen=max_len, padding='pre')
        pred = np.argmax(model.predict(t, verbose=0), axis=-1)[0]
        word = tok.index_word.get(pred, '')
        seed += ' ' + word
    return seed

In [72]:
print(predict_next("sherlock holmes was", 5))

sherlock holmes was a very good man who


In [73]:
test_sentences = [
    "to sherlock holmes she is always the",
    "i had seen little of holmes",
    "it is a capital mistake to",
    "you see but you do not",
    "my dear",
    "what do you make of",
    "holmes took up the",
    "i have no doubt that",
    "the man with the",
    "there is nothing more deceptive than an",
    "i think that you",
    "he threw himself down upon the",
    "we shall see what",
    "it was a cold",
    "mr holmes i",
]

for s in test_sentences:
    print(f"{s!r:50} -> {predict_next(s, 5)}")

'to sherlock holmes she is always the'             -> to sherlock holmes she is always the same peculiar tint of the
'i had seen little of holmes'                      -> i had seen little of holmes and i have been able
'it is a capital mistake to'                       -> it is a capital mistake to be a very good deal
'you see but you do not'                           -> you see but you do not know what you have been
'my dear'                                          -> my dear young lady has been been
'what do you make of'                              -> what do you make of the matter of the matter
'holmes took up the'                               -> holmes took up the door and i have been
'i have no doubt that'                             -> i have no doubt that i have been able to
'the man with the'                                 -> the man with the man who was a man
'there is nothing more deceptive than an'          -> there is nothing more deceptive than an hour of the house of